# MORTRA Mario on T4
GPU validation and persistent learning. Existing notebooks and memories are not overwritten. The player is based on c8bcb8cc4490271d599b0fc7beb61e665510253a. CUDA handles dense RGB statistics and FFT; the Java game and graph planner remain on CPU.

In [ ]:
import subprocess, sys, torch
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
assert 'T4' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
print('T4_READY', torch.cuda.get_device_name(0), 'torch', torch.__version__, 'CUDA', torch.version.cuda)


In [ ]:
from pathlib import Path
import os, subprocess, sys
SHA = 'e6ac5be605943add20deec0bcf51d3bad5e0e1c7'
REPO = Path('/content/mortra_t4_' + SHA[:8])
if not REPO.exists():
    subprocess.run(['git','clone','--filter=blob:none','--no-checkout','--depth=1','--branch','research/mario-fast-decisions-20260929','https://github.com/corcondor/mortra.git',str(REPO)],check=True)
    subprocess.run(['git','-C',str(REPO),'sparse-checkout','set','experiments/mario_continual','experiments/continual_tools','experiments/task_agent','experiments/noisy_rgb_discovery','experiments/noisy_rgb_version_space','tests'],check=True)
    subprocess.run(['git','-C',str(REPO),'fetch','--depth=1','origin',SHA],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',SHA],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == SHA
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jdk','xvfb','xauth'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','scipy','Pillow','psutil','pytest'],check=True)
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ['PATH']
os.environ.update(OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', MORTRA_DEVICE='cuda')
print('SOURCE_READY', SHA)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/MORTRA/mario-t4-e6ac5be6')


In [ ]:
def run_colab(mode):
    command = [sys.executable,'-u','-m','experiments.mario_continual.colab_t4',mode,'--output',str(OUT)]
    with subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        print(mode.upper(), 'PID', process.pid, flush=True)
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('Runner stopped; see the error above')
run_colab('prepare')


In [ ]:
run_colab('learn')
